# Experiment 2: Weakly Supervised Text Classification

**Paper**: Engineering Truthiness: A Standard for Pseudo–Ground Truth in Machine Learning Evaluation

**Description**: Real weak supervision regime using IMDB reviews with Snorkel labeling functions. Demonstrates a low-SRI regime where silver-based model selection is stable.

**Requirements**: See `requirements.txt` in the repository root.

**Usage**: Run cells sequentially. All outputs are saved to `outputs/{exp_name}/`.

---

In [ ]:
# =========================
# CONFIGURATION
# =========================
# Public mode: all data is generated synthetically or loaded from public sources.
# To use private data, create config/config_paths_private.py (gitignored).

import os, sys
from pathlib import Path

# Detect environment
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Optional: mount Google Drive for private data
    # from google.colab import drive
    # drive.mount('/content/drive')
    pass

# Resolve repo root (works from notebooks/ or repo root)
_nb_dir = Path('.').resolve()
_repo_root = _nb_dir.parent if _nb_dir.name == 'notebooks' else _nb_dir

# Try private config first, fall back to public defaults
try:
    sys.path.insert(0, str(_repo_root / 'config'))
    from config_paths_private import *  # noqa: F403
    CONFIG_MODE = 'private'
    print('[INFO] Using private path configuration')
except ImportError:
    # Public defaults — everything runs with synthetic data
    REPO_ROOT = _repo_root
    RUN_ID = 'public_run'
    DATADIR = REPO_ROOT / 'data'
    OUTDIR = REPO_ROOT / 'outputs'
    CONFIG_MODE = 'public'
    print('[INFO] Using public path configuration')

Path(OUTDIR).mkdir(parents=True, exist_ok=True)


In [ ]:
# =========================
# SHARED UTILITIES
# =========================
import os, json, time, platform, hashlib, random
from pathlib import Path
import numpy as np
import pandas as pd

def set_global_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass

def now_utc_compact():
    import datetime
    return datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S_utc")

def safe_mkdir(p: Path):
    p.mkdir(parents=True, exist_ok=True)
    return p

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def bootstrap_ci(x, n=1000, alpha=0.05, seed=42):
    rng = np.random.default_rng(seed)
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return (np.nan, np.nan, np.nan)
    stats = []
    for _ in range(n):
        samp = rng.choice(x, size=len(x), replace=True)
        stats.append(np.mean(samp))
    lo = np.quantile(stats, alpha/2)
    hi = np.quantile(stats, 1 - alpha/2)
    return (float(np.mean(x)), float(lo), float(hi))

def make_run_dirs(exp_name: str):
    run_id = f"{exp_name}_{now_utc_compact()}"
    base = Path("outputs") / exp_name / run_id
    figs = base / "figures"
    tabs = base / "tables"
    logs = base / "logs"
    for d in [base, figs, tabs, logs]:
        safe_mkdir(d)
    return run_id, base, figs, tabs, logs

def write_manifest(base: Path, meta: dict):
    mf = base / "run_manifest.json"
    meta = dict(meta)
    meta["platform"] = {
        "python": platform.python_version(),
        "platform": platform.platform()
    }
    mf.write_text(json.dumps(meta, indent=2))
    return mf

set_global_seed(42)


In [ ]:
# =========================
# 02 — SETUP + LOAD IMDB (PUBLIC)
# =========================
EXP = "02_text"
run_id, out_base, out_figs, out_tabs, out_logs = make_run_dirs(EXP)

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Try loading real IMDB dataset; fall back to synthetic if dependencies unavailable
try:
    from datasets import load_dataset
    ds = load_dataset("imdb")
    train_text = ds["train"]["text"][:8000]
    train_y_gold = np.array(ds["train"]["label"][:8000])
    test_text = ds["test"]["text"][:2000]
    test_y_gold = np.array(ds["test"]["label"][:2000])
    USE_REAL_IMDB = True
    print(f"[INFO] Loaded real IMDB: {len(train_text)} train, {len(test_text)} test")

except (ImportError, OSError) as e:
    # Synthetic fallback: generate text-like features for the ranking stability analysis
    print(f"[INFO] datasets library unavailable ({type(e).__name__}), using synthetic text data")
    rng_text = np.random.default_rng(42)
    
    pos_words = ["good", "great", "excellent", "wonderful", "amazing", "love", "best", "fantastic"]
    neg_words = ["bad", "terrible", "awful", "worst", "hate", "boring", "waste", "disappointing"]
    neutral_words = ["the", "movie", "film", "was", "it", "this", "a", "an", "with", "about", "story", "character"]
    
    def make_review(label, rng):
        words = rng.choice(neutral_words, size=rng.integers(20, 60)).tolist()
        signal = pos_words if label == 1 else neg_words
        n_signal = rng.integers(2, 6)
        for w in rng.choice(signal, size=n_signal):
            words.insert(rng.integers(0, len(words)), w)
        return " ".join(words)
    
    n_train, n_test = 8000, 2000
    train_y_gold = rng_text.integers(0, 2, size=n_train)
    test_y_gold = rng_text.integers(0, 2, size=n_test)
    train_text = [make_review(y, rng_text) for y in train_y_gold]
    test_text = [make_review(y, rng_text) for y in test_y_gold]
    USE_REAL_IMDB = False
    print(f"[INFO] Synthetic text data: {n_train} train, {n_test} test")

len(train_text), len(test_text)


In [ ]:
# =========================
# 02 — WEAK SUPERVISION (SILVER LABELS)
# =========================
ABSTAIN = -1
POS = 1
NEG = 0

try:
    from snorkel.labeling import LabelingFunction, PandasLFApplier
    from snorkel.labeling.model import LabelModel
    USE_SNORKEL = True
except ImportError:
    USE_SNORKEL = False
    print("[INFO] Snorkel unavailable, using simple keyword-based silver labels")

df_train = pd.DataFrame({"text": train_text, "gold": train_y_gold})
df_test = pd.DataFrame({"text": test_text, "gold": test_y_gold})

if USE_SNORKEL:
    def lf_contains_good(x):
        return POS if "good" in x.text.lower() else ABSTAIN
    def lf_contains_bad(x):
        return NEG if "bad" in x.text.lower() else ABSTAIN
    def lf_contains_excellent(x):
        return POS if "excellent" in x.text.lower() else ABSTAIN
    def lf_contains_waste(x):
        return NEG if "waste of time" in x.text.lower() else ABSTAIN

    lfs = [
        LabelingFunction(name="good", f=lf_contains_good),
        LabelingFunction(name="bad", f=lf_contains_bad),
        LabelingFunction(name="excellent", f=lf_contains_excellent),
        LabelingFunction(name="waste", f=lf_contains_waste),
    ]

    applier = PandasLFApplier(lfs=lfs)
    L_train = applier.apply(df=df_train)

    label_model = LabelModel(cardinality=2, verbose=False)
    label_model.fit(L_train=L_train, n_epochs=200, lr=0.01, seed=42)

    probs_train = label_model.predict_proba(L=L_train)
    y_train_silver = (probs_train[:,1] >= 0.5).astype(int)
    coverage = (L_train != ABSTAIN).any(axis=1).mean()
else:
    # Simple keyword voting as fallback silver labeler
    def keyword_silver_label(text):
        text_lower = text.lower()
        pos_score = sum(1 for w in ["good", "great", "excellent", "love", "best", "amazing"] if w in text_lower)
        neg_score = sum(1 for w in ["bad", "terrible", "awful", "worst", "hate", "boring"] if w in text_lower)
        if pos_score > neg_score:
            return 1
        elif neg_score > pos_score:
            return 0
        else:
            return np.random.RandomState(hash(text) % 2**31).randint(0, 2)
    
    y_train_silver = np.array([keyword_silver_label(t) for t in train_text])
    coverage = np.mean([1 for t in train_text if any(w in t.lower() for w in ["good","great","bad","terrible"])])

silver_pos_rate = y_train_silver.mean()
print(f"Coverage: {coverage:.3f}, Silver positive rate: {silver_pos_rate:.3f}")


In [ ]:
# =========================
# 02 — TRAIN MODELS + RANKING FLIPS
# =========================
Xtr, Xva, yg_tr, yg_va, ys_tr, ys_va = train_test_split(
    df_train["text"].values, df_train["gold"].values, y_train_silver,
    test_size=0.25, random_state=42, stratify=df_train["gold"].values
)

def fit_eval(C, use_silver_eval: bool):
    vec = TfidfVectorizer(max_features=40000, ngram_range=(1,2), stop_words="english")
    Xtrv = vec.fit_transform(Xtr)
    Xvav = vec.transform(Xva)
    Xtst = vec.transform(df_test["text"].values)

    clf = LogisticRegression(max_iter=300, C=C)
    clf.fit(Xtrv, ys_tr)  # train on SILVER (this is the point)

    # evaluation metric: gold vs silver
    pred_va = clf.predict(Xvav)
    pred_test = clf.predict(Xtst)

    score_va = accuracy_score(ys_va if use_silver_eval else yg_va, pred_va)
    score_test_gold = accuracy_score(df_test["gold"].values, pred_test)
    score_test_silverproxy = accuracy_score((pred_test), pred_test)  # placeholder not used

    return score_va, score_test_gold

Cs = [0.25, 0.5, 1.0, 2.0, 4.0]
gold_scores = []
silver_scores = []
test_gold_scores = []

for C in Cs:
    s_va, s_test_gold = fit_eval(C, use_silver_eval=True)
    g_va, _ = fit_eval(C, use_silver_eval=False)
    silver_scores.append(s_va)
    gold_scores.append(g_va)
    test_gold_scores.append(s_test_gold)

# pairwise flips between silver-eval ranking and gold-eval ranking
flips = 0
total = 0
for i in range(len(Cs)):
    for j in range(i+1, len(Cs)):
        total += 1
        s_ord = np.sign(silver_scores[i] - silver_scores[j])
        g_ord = np.sign(gold_scores[i] - gold_scores[j])
        if s_ord != g_ord:
            flips += 1

flip_rate = flips / total

df02 = pd.DataFrame({
    "C": Cs,
    "val_acc_silver_eval": silver_scores,
    "val_acc_gold_eval": gold_scores,
    "test_acc_gold": test_gold_scores
})
df02, flip_rate


In [ ]:
# =========================
# 02 — SAVE ARTIFACTS
# =========================
df02.to_csv(out_tabs / "results_02_text.csv", index=False)

import matplotlib.pyplot as plt
plt.figure(figsize=(7,4))
plt.plot(df02["C"], df02["val_acc_silver_eval"], marker="o", label="Val accuracy (silver-eval)")
plt.plot(df02["C"], df02["val_acc_gold_eval"], marker="x", linestyle="--", label="Val accuracy (gold-eval)")
plt.xscale("log")
plt.xlabel("Logistic Regression C (log scale)")
plt.ylabel("Validation accuracy")
plt.title(f"Ranking instability (flip rate = {flip_rate:.1%})")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(out_figs / "fig01_text_flip_instability.png", dpi=300)

manifest = write_manifest(out_base, {
    "experiment": EXP,
    "run_id": run_id,
    "imdb_train_n": len(train_text),
    "imdb_test_n": len(test_text),
    "coverage": float(coverage),
    "silver_pos_rate": float(silver_pos_rate),
    "Cs": Cs,
    "pairwise_flip_rate": float(flip_rate),
    "outputs": {
        "tables": [str(out_tabs / "results_02_text.csv")],
        "figures": [str(out_figs / "fig01_text_flip_instability.png")]
    }
})
manifest


---
## Appendix Figures: Experiment 2 (Weakly Supervised Text Classification)
The following figures are designed for direct inclusion in the paper appendix.

In [ ]:
# =========================
# APPENDIX FIG B1: Silver Label Quality - Agreement with Gold
# =========================
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(train_y_gold, y_train_silver)
silver_accuracy = np.trace(cm) / cm.sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
ax = axes[0]
im = ax.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm[i,j]:,}', ha='center', va='center', fontsize=14,
                color='white' if cm[i,j] > cm.max()/2 else 'black')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Negative', 'Positive'])
ax.set_yticklabels(['Negative', 'Positive'])
ax.set_xlabel('Silver Label', fontsize=12)
ax.set_ylabel('Gold Label', fontsize=12)
ax.set_title(f'Silver vs Gold Label Agreement\nAccuracy: {silver_accuracy:.1%}', fontsize=12)
plt.colorbar(im, ax=ax, shrink=0.8)

# Label distribution comparison
ax = axes[1]
x_pos = np.arange(2)
width = 0.35
gold_counts = np.bincount(train_y_gold, minlength=2)
silver_counts = np.bincount(y_train_silver, minlength=2)
ax.bar(x_pos - width/2, gold_counts, width, label='Gold Labels', color='#3498db', alpha=0.8)
ax.bar(x_pos + width/2, silver_counts, width, label='Silver Labels', color='#e74c3c', alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(['Negative', 'Positive'])
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Label Distribution: Gold vs Silver', fontsize=12)
ax.legend(fontsize=11)

fig.suptitle('Figure B1. Silver Label Quality Assessment (IMDB Weak Supervision)\n'
             'Left: Confusion matrix of silver vs gold labels on training data.\n'
             'Right: Class distribution shift introduced by the silver labeling process.',
             fontsize=11, y=1.06)
plt.tight_layout()
plt.savefig(out_figs / 'figB1_silver_label_quality.png', dpi=300, bbox_inches='tight')
plt.show()
print('[SAVED] figB1_silver_label_quality.png')


In [ ]:
# =========================
# APPENDIX FIG B2: Model Selection Divergence - Silver vs Gold Ranking
# =========================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Silver-eval ranking
ax = axes[0]
order_silver = np.argsort(df02['val_acc_silver_eval'].values)[::-1]
bars_s = ax.barh(range(len(Cs)), df02['val_acc_silver_eval'].values[order_silver],
        color='#3498db', alpha=0.8)
ax.set_yticks(range(len(Cs)))
ax.set_yticklabels([f'C={Cs[i]}' for i in order_silver])
ax.set_xlabel('Validation Accuracy (Silver Eval)', fontsize=11)
ax.set_title('Model Ranking Under Silver Evaluation', fontsize=12)
for i, idx in enumerate(order_silver):
    val = df02['val_acc_silver_eval'].values[idx]
    ax.text(val - 0.02, i, f'{val:.3f}', ha='right', va='center', fontsize=10,
            color='white', fontweight='bold')

# Gold-eval ranking
ax = axes[1]
order_gold = np.argsort(df02['val_acc_gold_eval'].values)[::-1]
bars_g = ax.barh(range(len(Cs)), df02['val_acc_gold_eval'].values[order_gold],
        color='#e74c3c', alpha=0.8)
ax.set_yticks(range(len(Cs)))
ax.set_yticklabels([f'C={Cs[i]}' for i in order_gold])
ax.set_xlabel('Validation Accuracy (Gold Eval)', fontsize=11)
ax.set_title('Model Ranking Under Gold Evaluation', fontsize=12)
for i, idx in enumerate(order_gold):
    val = df02['val_acc_gold_eval'].values[idx]
    ax.text(val - 0.005, i, f'{val:.3f}', ha='right', va='center', fontsize=10,
            color='white', fontweight='bold')

fig.suptitle('Figure B2. Model Selection Divergence: Silver vs Gold Evaluation Ranking\n'
             f'Pairwise ranking flip rate: {flip_rate:.1%}. '
             'Order differences indicate model selection instability under proxy labels.',
             fontsize=11, y=1.04)
plt.tight_layout()
plt.savefig(out_figs / 'figB2_model_selection_divergence.png', dpi=300, bbox_inches='tight')
plt.show()
print('[SAVED] figB2_model_selection_divergence.png')


In [ ]:
# =========================
# APPENDIX FIG B3: Three-Way Comparison
# =========================
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(df02['C'], df02['test_acc_gold'], marker='s', linewidth=2, color='#2ecc71',
        markersize=8, label='Test accuracy (gold labels)')
ax.plot(df02['C'], df02['val_acc_silver_eval'], marker='o', linewidth=2, color='#3498db',
        linestyle='--', markersize=8, label='Val accuracy (silver eval)')
ax.plot(df02['C'], df02['val_acc_gold_eval'], marker='x', linewidth=2, color='#e74c3c',
        linestyle=':', markersize=8, label='Val accuracy (gold eval)')

best_silver_C = df02.loc[df02['val_acc_silver_eval'].idxmax(), 'C']
best_gold_C = df02.loc[df02['val_acc_gold_eval'].idxmax(), 'C']
ax.axvline(best_silver_C, color='#3498db', alpha=0.3, linestyle='--', label=f'Silver best: C={best_silver_C}')
ax.axvline(best_gold_C, color='#e74c3c', alpha=0.3, linestyle=':', label=f'Gold best: C={best_gold_C}')

ax.set_xscale('log')
ax.set_xlabel('Logistic Regression C (log scale)', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Figure B3. Three-Way Comparison: Silver Eval, Gold Eval, and Gold Test Performance\n'
             'Shows whether silver-based model selection yields competitive ground truth performance.',
             fontsize=11, pad=15)
ax.legend(fontsize=9, loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(out_figs / 'figB3_three_way_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print('[SAVED] figB3_three_way_comparison.png')


In [ ]:
# =========================
# APPENDIX TABLE B1: Full Results Summary
# =========================
print('Table B1. IMDB Weak Supervision: Model Performance Under Silver and Gold Evaluation')
print('=' * 80)
print(f'{"C":>6s}  {"Silver Eval":>12s}  {"Gold Eval":>12s}  {"Gold Test":>12s}  {"Silver Pick":>12s}  {"Gold Pick":>10s}')
print('-' * 80)
best_s = df02['val_acc_silver_eval'].idxmax()
best_g = df02['val_acc_gold_eval'].idxmax()
for idx, row in df02.iterrows():
    s_flag = '  <--' if idx == best_s else ''
    g_flag = '  <--' if idx == best_g else ''
    print(f'{row["C"]:>6.2f}  {row["val_acc_silver_eval"]:>12.4f}  {row["val_acc_gold_eval"]:>12.4f}  '
          f'{row["test_acc_gold"]:>12.4f}  {s_flag:>12s}  {g_flag:>10s}')
print('=' * 80)
print(f'Pairwise flip rate: {flip_rate:.1%}')
print(f'Coverage: {coverage:.3f} | Silver positive rate: {silver_pos_rate:.3f}')
data_source = 'Real IMDB (HuggingFace)' if USE_REAL_IMDB else 'Synthetic text'
label_source = 'Snorkel LabelModel' if USE_SNORKEL else 'Keyword voting'
print(f'Data: {data_source} | Silver labeler: {label_source}')


In [ ]:
from IPython.display import Image, display as ipy_display
import glob

print("Saved figures:")
for p in sorted(glob.glob(str(out_figs / "*.png"))):
    print(" -", p)
    ipy_display(Image(filename=p))


In [ ]:
# ==============================
# END OF NOTEBOOK SUMMARY
# ==============================
import json
summary = {
    "run_dir": str(out_base),
    "fig_dir": str(out_figs),
    "tables_dir": str(out_tabs),
}
summary_path = out_base / "run_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print("[DONE] Outputs saved under:", out_base)
